In [0]:
%sql
USE CATALOG retail_dwh;
USE DATABASE gold;

In [0]:
%sql
SELECT 'DimCustomer Total'    AS table_name, COUNT(*) AS row_count FROM retail_dwh.gold.DimCustomer
UNION ALL
SELECT 'DimCustomer Active',   COUNT(*) FROM retail_dwh.gold.DimCustomer WHERE IsActive = 1
UNION ALL
SELECT 'DimCustomer Expired',  COUNT(*) FROM retail_dwh.gold.DimCustomer WHERE IsActive = 0
UNION ALL
SELECT 'DimProduct',           COUNT(*) FROM retail_dwh.gold.DimProduct
UNION ALL
SELECT 'DimStore',             COUNT(*) FROM retail_dwh.gold.DimStore
UNION ALL
SELECT 'FactSales',            COUNT(*) FROM retail_dwh.gold.FactSales
UNION ALL
SELECT 'Monthly Summary',      COUNT(*) FROM retail_dwh.gold.monthly_sales_summary
UNION ALL
SELECT 'Region Summary',       COUNT(*) FROM retail_dwh.gold.region_sales_summary
UNION ALL
SELECT 'Top Products',         COUNT(*) FROM retail_dwh.gold.top_products_summary;

In [0]:
%sql
SELECT COUNT(*) AS multiple_active_record_count
FROM (
    SELECT CustomerID
    FROM retail_dwh.gold.DimCustomer
    WHERE IsActive = 1
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
);

In [0]:
%sql

SELECT COUNT(*) AS expired_without_enddate_count
FROM retail_dwh.gold.DimCustomer
WHERE IsActive = 0
AND EndDate IS NULL;

In [0]:
%sql

SELECT COUNT(*) AS invalid_date_range_count
FROM retail_dwh.gold.DimCustomer
WHERE EndDate IS NOT NULL
AND StartDate > EndDate;

In [0]:
%sql

SELECT COUNT(*) AS null_sk_count
FROM retail_dwh.gold.FactSales
WHERE CustomerSK IS NULL
   OR ProductSK IS NULL
   OR StoreSK IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS orphan_customerkey_count
FROM (
    SELECT f.CustomerSK
    FROM retail_dwh.gold.FactSales f
    LEFT JOIN retail_dwh.gold.DimCustomer c
    ON f.CustomerSK = c.CustomerSK
    WHERE c.CustomerSK IS NULL
);

In [0]:
%sql
SELECT COUNT(*) AS orphan_productkey_count
FROM (
    SELECT f.ProductSK
    FROM retail_dwh.gold.FactSales f
    LEFT JOIN retail_dwh.gold.DimProduct p
    ON f.ProductSK = p.ProductSK
    WHERE p.ProductSK IS NULL
);

In [0]:
%sql

SELECT COUNT(*) AS orphan_storekey_count
FROM (
    SELECT f.StoreSK
    FROM retail_dwh.gold.FactSales f
    LEFT JOIN retail_dwh.gold.DimStore s
    ON f.StoreSK = s.StoreSK
    WHERE s.StoreSK IS NULL
);

In [0]:
%sql

SELECT COUNT(*) AS incorrect_amount_count
FROM (
    SELECT
        f.TransactionID
    FROM retail_dwh.gold.FactSales f
    JOIN retail_dwh.gold.DimProduct p
    ON f.ProductSK = p.ProductSK
    WHERE f.Amount != ROUND(f.Quantity * p.UnitPrice, 2)
);

In [0]:
%sql

SELECT COUNT(*) AS duplicate_fact_transaction_count
FROM (
    SELECT TransactionID
    FROM retail_dwh.gold.FactSales
    GROUP BY TransactionID
    HAVING COUNT(*) > 1
);